# OncoRisk · Cuaderno de análisis exploratorio

**Informe 2, sección VI-D.4 (Análisis exploratorio) y arquitectura VII-B.4.**
Este cuaderno revisa la estructura, la calidad y la distribución del conjunto de datos antes del modelado:
valores faltantes, distribución de las variables, balance de clases y posibles fugas de información.

- **Fuente:** `data/synthetic/pacientes_sinteticos_20000.csv`, el mismo lote que se cargó a Snowflake por Stage
  (`RAW.PACIENTES_PRUEBA_SINTETICOS`). Es **sintético**: no contiene pacientes del hospital.
- **Alcance:** descriptivo. Las asociaciones que aparecen aquí **no son relaciones causales**; además, las relaciones
  entre variables y etiqueta las impuso el generador sintético.
- Las decisiones tomadas a partir de este análisis se resumen al final.

Para ejecutarlo: `python -m pip install -r requirements-notebook.txt` y luego `python -m jupyter lab` desde la raíz del proyecto.

In [1]:
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from config import POST_OUTCOME_LEAKY_COLUMNS, TARGET_COLUMN
from src.analytics.data import load_patient_data
from src.processing.feature_engineering import CATEGORICAL_FEATURES, NON_FEATURE_COLUMNS, NUMERIC_FEATURES

pd.set_option("display.precision", 3)
df, meta = load_patient_data(ROOT / "data" / "synthetic" / "pacientes_sinteticos_20000.csv")
print(f"Archivo: {meta['file_name']}  ·  SHA-256: {meta['hash'][:16]}…")
print(f"{meta['rows']:,} filas  ·  {meta['patients']:,} pacientes únicos  ·  {meta['classification']}")

Archivo: pacientes_sinteticos_20000.csv  ·  SHA-256: bf2078859176c229…
20,000 filas  ·  20,000 pacientes únicos  ·  Sintética: generador del prototipo, semilla 42


## 1. Estructura del conjunto

In [2]:
estructura = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "valores_unicos": df.nunique(),
    "rol": ["objetivo" if c == TARGET_COLUMN else "metadato / no predictor" if c in NON_FEATURE_COLUMNS
            else "derivada" if c in ("imc_categoria", "grupo_etario") else "predictor" for c in df.columns],
})
estructura

,tipo,no_nulos,valores_unicos,rol
id_paciente,string,20000,20000,metadato / no predictor
edad,float64,20000,741,predictor
sexo,string,20000,2,predictor
antecedente_familiar_cancer,int64,20000,2,predictor
antecedente_personal_benigno,int64,20000,2,predictor
tabaquismo,string,20000,3,predictor
imc,float64,20000,290,predictor
presion_sistolica,float64,20000,100,predictor
glucosa_mg_dl,float64,20000,124,predictor
paridad,int64,20000,9,predictor


In [3]:
df.head()

,id_paciente,edad,sexo,antecedente_familiar_cancer,antecedente_personal_benigno,tabaquismo,imc,presion_sistolica,glucosa_mg_dl,paridad,ca125_u_ml,resultado_imagen,fuente,periodo,riesgo_oncologico,imc_categoria,grupo_etario
0,PAC-00001,59.0,F,0,0,exfumador,35.2,129.0,78.0,1,11.11,sospechoso,consulta_externa,2023-S1,1,obesidad,55-69
1,PAC-00002,41.5,F,0,0,activo,25.4,133.0,56.0,2,11.59,normal,consulta_externa,2024-S2,0,sobrepeso,40-54
2,PAC-00003,64.8,F,0,0,nunca,18.7,117.0,124.0,2,14.25,normal,consulta_externa,2024-S1,0,normal,55-69
3,PAC-00004,67.2,M,0,1,nunca,25.8,136.0,117.0,1,7.98,normal,urgencias,2024-S2,0,sobrepeso,55-69
4,PAC-00005,29.6,F,0,0,nunca,35.3,142.0,120.0,2,11.61,normal,consulta_externa,2024-S2,0,obesidad,<40


## 2. Calidad de los datos
Criterios del informe (VI-D.2): completitud, validez, consistencia, unicidad y valores extremos.

In [4]:
faltantes = df.isna().sum().to_frame("faltantes")
faltantes["porcentaje"] = 100 * faltantes["faltantes"] / len(df)
print("Filas duplicadas exactas:", int(df.duplicated().sum()))
print("Identificadores repetidos:", int(df["id_paciente"].duplicated().sum()))
print("Coerciones numéricas fallidas:", meta["numeric_coercions"])
faltantes.query("faltantes > 0") if faltantes["faltantes"].any() else print("No hay valores faltantes.")

Filas duplicadas exactas: 0
Identificadores repetidos: 0
Coerciones numéricas fallidas: {'edad': 0, 'imc': 0, 'presion_sistolica': 0, 'glucosa_mg_dl': 0, 'paridad': 0, 'ca125_u_ml': 0, 'antecedente_familiar_cancer': 0, 'antecedente_personal_benigno': 0, 'riesgo_oncologico': 0}
No hay valores faltantes.


In [5]:
# Validez: reglas de dominio que aplica la capa CLEAN (sql/02_raw_a_clean.sql)
reglas = {
    "edad fuera de [0, 120]": ~df["edad"].between(0, 120) & df["edad"].notna(),
    "IMC fuera de [10, 80]": ~df["imc"].between(10, 80) & df["imc"].notna(),
    "objetivo faltante": df[TARGET_COLUMN].isna(),
}
pd.Series({regla: int(mascara.sum()) for regla, mascara in reglas.items()}, name="registros")

edad fuera de [0, 120]    0
IMC fuera de [10, 80]     0
objetivo faltante         0
Name: registros, dtype: int64

In [6]:
# Valores extremos por la regla de Tukey (fuera de Q1 − 1,5·IQR o Q3 + 1,5·IQR)
filas = []
for col in NUMERIC_FEATURES:
    x = df[col].dropna()
    q1, q3 = x.quantile([.25, .75])
    iqr = q3 - q1
    fuera = ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()
    filas.append({"variable": col, "mínimo": x.min(), "Q1": q1, "mediana": x.median(), "Q3": q3,
                  "máximo": x.max(), "atípicos_IQR": int(fuera), "atípicos_%": 100 * fuera / len(x)})
extremos = pd.DataFrame(filas).set_index("variable")
extremos

,mínimo,Q1,mediana,Q3,máximo,atípicos_IQR,atípicos_%
variable,,,,,,,
edad,18.0,46.30,55.00,63.80,95.00,147,0.735
imc,15.0,23.60,26.90,30.40,47.20,80,0.400
presion_sistolica,85.0,114.00,125.00,136.00,198.00,45,0.225
glucosa_mg_dl,55.0,85.00,100.00,115.00,188.00,58,0.290
paridad,0.0,1.00,1.00,2.00,8.00,1548,7.740
ca125_u_ml,2.0,9.64,13.52,18.91,86.73,784,3.920


**Tratamiento de valores extremos.** Solo se eliminan los valores *imposibles* (reglas de arriba). Los atípicos por IQR
son valores clínicamente plausibles (por ejemplo, CA-125 alto), así que **se conservan**: eliminarlos quitaría justamente
los casos de mayor interés. Su efecto se controla con la imputación por mediana, el escalado estándar y modelos de
árboles, que no dependen de la escala. Esta decisión está documentada en `documentacion/decisiones_modelado.md`.

## 3. Variable objetivo y balance de clases

In [7]:
balance = df[TARGET_COLUMN].value_counts().rename_axis("clase").to_frame("pacientes")
balance["porcentaje"] = 100 * balance["pacientes"] / balance["pacientes"].sum()
print(f"Prevalencia de la clase 1: {df[TARGET_COLUMN].mean():.1%}")
balance

Prevalencia de la clase 1: 24.9%


,pacientes,porcentaje
clase,,
0,15012,75.06
1,4988,24.94


In [8]:
por_periodo = df.groupby("periodo")[TARGET_COLUMN].agg(pacientes="size", prevalencia="mean").reset_index()
alt.Chart(por_periodo).mark_bar(color="#168eb5").encode(
    x=alt.X("periodo:N", title="Semestre simulado"),
    y=alt.Y("prevalencia:Q", title="Proporción con etiqueta 1", axis=alt.Axis(format="%")),
    tooltip=["periodo", "pacientes", alt.Tooltip("prevalencia:Q", format=".1%")],
).properties(width=360, height=220, title="Prevalencia por semestre")

alt.Chart(...)

Las clases están desbalanceadas (aprox. 1 de cada 4 pacientes con etiqueta 1). Por eso la evaluación no se limita a la
exactitud: se usan AUROC, PR-AUC, sensibilidad, especificidad, F1 y Brier, y el umbral se elige con datos de desarrollo.

## 4. Distribución de las variables numéricas

In [9]:
df[NUMERIC_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
edad,20000.0,55.077,13.024,18.0,46.30,55.00,63.80,95.00
imc,20000.0,27.000,5.001,15.0,23.60,26.90,30.40,47.20
presion_sistolica,20000.0,125.082,15.967,85.0,114.00,125.00,136.00,198.00
glucosa_mg_dl,20000.0,100.098,21.628,55.0,85.00,100.00,115.00,188.00
paridad,20000.0,1.590,1.261,0.0,1.00,1.00,2.00,8.00
ca125_u_ml,20000.0,15.305,8.155,2.0,9.64,13.52,18.91,86.73


In [10]:
def histograma(col, bins=30):
    edges = np.histogram_bin_edges(df[col].dropna(), bins=bins)
    partes = []
    for clase, grupo in df.groupby(TARGET_COLUMN):
        conteo, _ = np.histogram(grupo[col].dropna(), bins=edges)
        partes.append(pd.DataFrame({"desde": edges[:-1], "hasta": edges[1:], "densidad": conteo / conteo.sum(),
                                    "clase": f"etiqueta {int(clase)}"}))
    datos = pd.concat(partes)
    return alt.Chart(datos).mark_bar(opacity=.55).encode(
        x=alt.X("desde:Q", title=col, bin="binned"), x2="hasta:Q",
        y=alt.Y("densidad:Q", title="Proporción", stack=None),
        color=alt.Color("clase:N", scale=alt.Scale(range=["#83939d", "#d0592b"]), title=None),
    ).properties(width=250, height=160)

alt.vconcat(*[alt.hconcat(*[histograma(c) for c in NUMERIC_FEATURES[i:i + 3]]) for i in (0, 3)])

alt.VConcatChart(...)

## 5. Variables categóricas frente a la etiqueta

In [11]:
tablas = []
for col in CATEGORICAL_FEATURES + ["imc_categoria", "grupo_etario"]:
    t = df.groupby(col, dropna=False)[TARGET_COLUMN].agg(pacientes="size", proporcion_etiqueta_1="mean").reset_index()
    t.insert(0, "variable", col)
    tablas.append(t.rename(columns={col: "categoria"}))
tasas = pd.concat(tablas, ignore_index=True)
tasas["categoria"] = tasas["categoria"].astype(str)
tasas

,variable,categoria,pacientes,proporcion_etiqueta_1
0,sexo,F,15583,0.250
1,sexo,M,4417,0.248
2,antecedente_familiar_cancer,0,15589,0.217
3,antecedente_familiar_cancer,1,4411,0.364
4,antecedente_personal_benigno,0,17578,0.243
5,antecedente_personal_benigno,1,2422,0.295
6,tabaquismo,activo,4051,0.323
7,tabaquismo,exfumador,5000,0.252
8,tabaquismo,nunca,10949,0.221
9,resultado_imagen,indeterminado,5193,0.309


In [12]:
alt.Chart(tasas).mark_bar(color="#168eb5").encode(
    y=alt.Y("categoria:N", title=None, sort=None),
    x=alt.X("proporcion_etiqueta_1:Q", title="Proporción con etiqueta 1", axis=alt.Axis(format="%")),
    tooltip=["variable", "categoria", "pacientes", alt.Tooltip("proporcion_etiqueta_1:Q", format=".1%")],
).properties(width=280, height=90).facet(facet=alt.Facet("variable:N", title=None), columns=3).resolve_scale(y="independent")

alt.FacetChart(...)

## 6. Correlaciones entre variables numéricas

In [13]:
corr = df[NUMERIC_FEATURES + [TARGET_COLUMN]].corr(method="spearman")
largo = corr.reset_index().melt(id_vars="index", var_name="variable_2", value_name="rho").rename(columns={"index": "variable_1"})
base = alt.Chart(largo).encode(x=alt.X("variable_1:N", title=None, sort=None), y=alt.Y("variable_2:N", title=None, sort=None))
(base.mark_rect().encode(color=alt.Color("rho:Q", scale=alt.Scale(scheme="redblue", domain=[-1, 1]), title="ρ Spearman"))
 + base.mark_text(fontSize=10).encode(text=alt.Text("rho:Q", format=".2f"))).properties(width=360, height=360)

alt.LayerChart(...)

Ninguna pareja de predictores muestra una correlación tan alta como para considerarlos redundantes, así que se
conservan todos. Una correlación con la etiqueta indica asociación en este lote sintético, no causa.

## 7. Fuga de información y temporalidad

In [14]:
presentes = [c for c in POST_OUTCOME_LEAKY_COLUMNS if c in df.columns]
print("Columnas posteriores al desenlace presentes en el archivo:", presentes or "ninguna")
print("Se excluyen del modelo como metadatos:", [c for c in NON_FEATURE_COLUMNS if c in df.columns])
print("Predictores usados:", NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["imc_categoria", "grupo_etario"])
print("Pacientes con más de una fila:", int(df["id_paciente"].duplicated().sum()))

Columnas posteriores al desenlace presentes en el archivo: ninguna
Se excluyen del modelo como metadatos: ['id_paciente', 'fuente', 'periodo', 'riesgo_oncologico']
Predictores usados: ['edad', 'imc', 'presion_sistolica', 'glucosa_mg_dl', 'paridad', 'ca125_u_ml', 'sexo', 'antecedente_familiar_cancer', 'antecedente_personal_benigno', 'tabaquismo', 'resultado_imagen', 'imc_categoria', 'grupo_etario']
Pacientes con más de una fila: 0


- `fuente` y `periodo` describen la procedencia del registro y **no** entran al modelo.
- No hay fecha índice ni seguimiento: según el marco conceptual del informe, la salida se interpreta como
  **probabilidad de clasificación de la condición registrada**, no como riesgo futuro.
- La partición entrenamiento/prueba y la validación cruzada se agrupan por `id_paciente`.

## 8. Decisiones adoptadas a partir del análisis

| Hallazgo | Decisión |
|---|---|
| Valores imposibles de edad o IMC | Se eliminan en CLEAN (`sql/02_raw_a_clean.sql`) |
| Atípicos plausibles (IQR) | Se conservan; imputación por mediana y escalado robustos |
| Faltantes en variables numéricas o categóricas | Se imputan dentro del pipeline, ajustado solo con desarrollo |
| Clases desbalanceadas (~25 % positivos) | Métricas más allá de exactitud; umbral elegido con desarrollo |
| Columnas posteriores al desenlace | Se excluyen antes del modelado |
| Sin temporalidad | La salida se llama probabilidad de clasificación, no riesgo |